In [1]:
import dask
import xarray as xr

In [2]:
VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds", "hurs"]

### Option 1: Calculate from scratch

In [9]:
def calculate_thresholds(obs_max, obs_min, obs_max_std, obs_min_std):
    outlier_thresh_high = obs_max + (5 * obs_max_std)
    outlier_thresh_low = obs_min - (5 * obs_min_std)

    return outlier_thresh_low, outlier_thresh_high

In [3]:
from srm import catalog

In [4]:
obs = catalog.get("ERA5").to_xarray()
obs = obs[VARIABLES]

In [5]:
from srm.downscaling_utils import rechunk

obs = xr.Dataset(
    {var: rechunk(da=obs[var], pattern="full_time") for var in obs.data_vars},
    attrs=obs.attrs,
)

In [6]:
def calculate_stats(obs=obs, subset=None, timescale="annual"):
    if subset is not None:
        obs_subset = obs.sel(lat=subset[0], lon=subset[1]).load()
    else:
        obs_subset = obs
    # obs_subset = obs_subset.chunk({'time': -1,'lat':1,'lon':1})

    if timescale == "annual":
        annual = obs_subset.groupby("time.year")
        obs_annual_max, obs_annual_min = dask.compute(annual.max(), annual.min())

        obs_max = obs_annual_max.max(dim=["year"])
        obs_min = obs_annual_min.min(dim=["year"])

        obs_max_std = obs_annual_max.std(dim=["year"])
        obs_min_std = obs_annual_min.std(dim=["year"])

    elif timescale == "monthly":
        monthly = obs_subset.groupby(["time.year", "time.month"])
        obs_monthly_max, obs_monthly_min = dask.compute(monthly.max(), monthly.min())

        obs_max = obs_monthly_max.max(dim="year")
        obs_min = obs_monthly_min.min(dim="year")
        obs_max_std = obs_monthly_max.std(dim="year")
        obs_min_std = obs_monthly_min.std(dim="year")

    elif timescale == "dayofyear":
        rolling_doy_max = obs_subset.rolling(time=30, center=True).max()
        rolling_doy_min = obs_subset.rolling(time=30, center=True).min()

        obs_max = rolling_doy_max.groupby("time.dayofyear").max()
        obs_min = rolling_doy_min.groupby("time.dayofyear").min()
        obs_max_std = rolling_doy_max.groupby("time.dayofyear").std()
        obs_min_std = rolling_doy_min.groupby("time.dayofyear").std()

    return [obs_max, obs_min, obs_max_std, obs_min_std]

##### Annual stats

In [7]:
[obs_max, obs_min, obs_max_std, obs_min_std] = calculate_stats(subset=None, timescale="annual")

In [ ]:
[outlier_thresh_low, outlier_thresh_high] = calculate_thresholds(
    obs_max, obs_min, obs_max_std, obs_min_std
)

In [27]:
combined = xr.merge(
    [
        obs_max.rename({v: f"{v}_max" for v in obs_max.data_vars}),
        obs_min.rename({v: f"{v}_min" for v in obs_min.data_vars}),
        obs_max_std.rename({v: f"{v}_max_std" for v in obs_max_std.data_vars}),
        obs_min_std.rename({v: f"{v}_min_std" for v in obs_min_std.data_vars}),
    ]
)

combined.to_zarr("s3://carbonplan-scratch/srm/qaqc/annual_obs_thresholds_global.zarr", mode="w")

/opt/coiled/env/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


##### Day of year stats

In [8]:
[obs_max, obs_min, obs_max_std, obs_min_std] = calculate_stats(subset=None, timescale="dayofyear")

In [11]:
STORE = "s3://carbonplan-scratch/srm/qaqc/doy_obs_thresholds_global.zarr"

stats = {"max": obs_max, "min": obs_min, "max_std": obs_max_std, "min_std": obs_min_std}

for i, var in enumerate(VARIABLES):
    per_var = xr.merge(
        [
            stats["max"][[var]].rename({var: f"{var}_max"}),
            stats["min"][[var]].rename({var: f"{var}_min"}),
            stats["max_std"][[var]].rename({var: f"{var}_max_std"}),
            stats["min_std"][[var]].rename({var: f"{var}_min_std"}),
        ]
    )
    mode = "w" if i == 0 else "a"
    per_var.to_zarr(STORE, mode=mode)

/opt/coiled/env/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
